# Advanced Python Dictionaries — Tutorial-Style Problems with Complete Solutions

This notebook develops advanced dictionary skills through a sequence of **new problems**.

The style deliberately follows a tutorial rhythm:

1. introduce a small idea,
2. inspect it with short code cells,
3. point out a subtle behavior,
4. state a problem,
5. solve the problem in logical steps,
6. verify the solution with examples and assertions.

The emphasis is not only on getting a dictionary to work. We will also make the construction rules explicit, preserve useful ordering, avoid accidental aliasing, select good key types, and build reusable indexing functions.

## Topics Covered

We will work with:

- strict construction from parallel iterables,
- generators and one-shot iterators,
- insertion order and stable deduplication,
- dynamic dictionary views,
- tuple and frozen-dataclass keys,
- bidirectional mappings,
- dictionaries whose values are lists,
- primary and secondary indexes,
- callable dispatch tables,
- copy-on-write updates,
- nested cross-tabulations,
- custom dictionaries with `__missing__`,
- hash joins,
- and a dictionary-driven rule engine.

All examples use only the Python standard library.

## Best-Practice Checklist

As you work through the problems, keep these questions in mind:

- Are the proposed keys hashable and stable?
- What happens when an input key appears more than once?
- Can two input sequences have different lengths?
- Does the function mutate caller-owned data?
- Are nested mutable objects accidentally shared?
- Is a comprehension still readable, or would an explicit loop be clearer?
- Is insertion order part of the required behavior?
- Are dictionary views being used as live views or as frozen snapshots?
- Can an index become inconsistent with the underlying records?
- Are error messages specific enough to diagnose invalid input?

## Setup

We will use a few standard-library tools throughout the notebook.

In [1]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from itertools import zip_longest
from typing import Any, Callable, Hashable, Iterable, Iterator, Mapping, Sequence
from urllib.parse import parse_qsl

In [2]:
def heading(title: str) -> None:
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")

In [3]:
heading("Notebook setup complete")


Notebook setup complete


# Problem 1 — Construct a Dictionary from Parallel Iterables Without Silent Data Loss

A common pattern is to combine one iterable of keys with another iterable of values.

In [4]:
keys = ["alpha", "beta", "gamma"]
values = [10, 20, 30]

ordinary = dict(zip(keys, values))
ordinary

{'alpha': 10, 'beta': 20, 'gamma': 30}

That works when both inputs contain the same number of elements.

But ordinary `zip` stops as soon as the shorter input is exhausted.

In [5]:
short_values = [10, 20]

silently_truncated = dict(zip(keys, short_values))
silently_truncated

{'alpha': 10, 'beta': 20}

The key `"gamma"` disappeared without an exception.

For data-loading code, silent truncation is often dangerous.

### Your task

Write `strict_dict(keys, values)` with these rules:

- construct a dictionary from parallel iterables,
- raise `ValueError` if the lengths differ,
- raise `ValueError` if a key occurs more than once,
- accept generators as well as lists,
- and avoid converting the entire inputs to lists merely to compare lengths.

## Step 1 — Detect Unequal Lengths

`itertools.zip_longest` continues until the longer iterable is exhausted.

We can insert a unique sentinel object. If the sentinel appears on only one side, the lengths differ.

In [6]:
_SENTINEL = object()

list(zip_longest(["a", "b"], [1], fillvalue=_SENTINEL))

[('a', 1), ('b', <object at 0x17b18641490>)]

## Step 2 — Build Incrementally

An explicit loop is clearer than a comprehension here because we need two validation checks.

In [7]:
def strict_dict(
    keys: Iterable[Hashable],
    values: Iterable[Any],
) -> dict[Hashable, Any]:
    result: dict[Hashable, Any] = {}

    for position, (key, value) in enumerate(
        zip_longest(keys, values, fillvalue=_SENTINEL)
    ):
        if key is _SENTINEL or value is _SENTINEL:
            raise ValueError(
                f"keys and values have different lengths near position {position}"
            )

        if key in result:
            raise ValueError(f"duplicate key at position {position}: {key!r}")

        result[key] = value

    return result

## Step 3 — Verify Normal, Unequal, Duplicate, and Generator Inputs

In [8]:
strict_dict(["a", "b", "c"], [1, 2, 3])

{'a': 1, 'b': 2, 'c': 3}

In [9]:
try:
    strict_dict(["a", "b", "c"], [1, 2])
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError keys and values have different lengths near position 2


In [10]:
try:
    strict_dict(["a", "b", "a"], [1, 2, 3])
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError duplicate key at position 2: 'a'


In [11]:
generated_keys = (f"k{i}" for i in range(4))
generated_values = (i * i for i in range(4))

result = strict_dict(generated_keys, generated_values)
result

{'k0': 0, 'k1': 1, 'k2': 4, 'k3': 9}

In [12]:
assert result == {"k0": 0, "k1": 1, "k2": 4, "k3": 9}

### Best-practice observation

Use `dict(zip(...))` when truncation is acceptable or lengths are already guaranteed.

Use a validated loop when length mismatches or duplicate keys represent bad data.

# Problem 2 — Understand One-Shot Iterators Passed to `dict`

The `dict` constructor accepts any iterable whose elements are two-item iterables.

In [13]:
pairs = [("x", 1), ("y", 2), ("z", 3)]
dict(pairs)

{'x': 1, 'y': 2, 'z': 3}

A generator also satisfies that requirement.

In [14]:
pair_generator = ((letter, index) for index, letter in enumerate("abc", start=1))
dict(pair_generator)

{'a': 1, 'b': 2, 'c': 3}

But a generator is a **one-shot iterator**.

After it has been consumed, asking for more items produces nothing.

In [15]:
list(pair_generator)

[]

### Your task

Write a reusable function `build_and_audit_pairs` that:

- accepts a one-shot iterable of pairs,
- constructs the dictionary,
- returns both the dictionary and an audit tuple containing the original pairs,
- validates that every item has exactly two elements,
- and reports the position of malformed input.

## Step 1 — Materialize Once, Deliberately

Because the same source must support both construction and auditing, materializing it once is appropriate.

In [16]:
def build_and_audit_pairs(
    pairs: Iterable[Iterable[Any]],
) -> tuple[dict[Any, Any], tuple[tuple[Any, Any], ...]]:
    audited: list[tuple[Any, Any]] = []

    for position, item in enumerate(pairs):
        unpacked = tuple(item)

        if len(unpacked) != 2:
            raise ValueError(
                f"item at position {position} must contain exactly two values; "
                f"received {len(unpacked)}"
            )

        key, value = unpacked
        audited.append((key, value))

    audit_tuple = tuple(audited)
    return dict(audit_tuple), audit_tuple

## Step 2 — Use a Generator and Preserve Its Consumed Data

In [17]:
source = ((name, len(name)) for name in ["Ada", "Grace", "Linus"])

name_lengths, audit = build_and_audit_pairs(source)

name_lengths, audit

({'Ada': 3, 'Grace': 5, 'Linus': 5}, (('Ada', 3), ('Grace', 5), ('Linus', 5)))

In [18]:
assert name_lengths == {"Ada": 3, "Grace": 5, "Linus": 5}
assert audit == (("Ada", 3), ("Grace", 5), ("Linus", 5))
assert list(source) == []

## Step 3 — Diagnose a Malformed Element

In [19]:
bad_source = [("a", 1), ("b", 2, "extra"), ("c", 3)]

try:
    build_and_audit_pairs(bad_source)
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError item at position 1 must contain exactly two values; received 3


### Best-practice observation

Do not materialize an iterable automatically.

Materialize it when you genuinely need replay, auditing, sorting, length inspection, or multiple passes.

# Problem 3 — Perform Stable Deduplication with Explicit First-Wins and Last-Wins Rules

Dictionaries preserve insertion order.

That makes them useful for stable deduplication, but the result depends on the update policy.

In [20]:
words = ["pear", "apple", "pear", "banana", "apple", "kiwi"]

Constructing from the sequence as keys keeps the first insertion position of each distinct key.

In [21]:
list(dict.fromkeys(words))

['pear', 'apple', 'banana', 'kiwi']

However, suppose each occurrence also has a score.

Repeated assignment changes the value but does not move the existing key to the end.

In [22]:
observations = [
    ("pear", 2),
    ("apple", 5),
    ("pear", 7),
    ("banana", 3),
    ("apple", 9),
    ("kiwi", 4),
]

last_value_by_first_position = dict(observations)
last_value_by_first_position

{'pear': 7, 'apple': 9, 'banana': 3, 'kiwi': 4}

Notice the mixed behavior:

- key positions are based on first insertion,
- values come from the last assignment.

### Your task

Implement two functions:

- `first_wins(pairs)` keeps the first value and first position,
- `last_wins_and_moves(pairs)` keeps the last value and orders keys by their last occurrence.

## Step 1 — First-Wins Construction

`setdefault` inserts only when the key is absent.

In [23]:
def first_wins(pairs: Iterable[tuple[Hashable, Any]]) -> dict[Hashable, Any]:
    result: dict[Hashable, Any] = {}

    for key, value in pairs:
        result.setdefault(key, value)

    return result

In [24]:
first_wins(observations)

{'pear': 2, 'apple': 5, 'banana': 3, 'kiwi': 4}

## Step 2 — Last-Wins and Move-to-End Construction

Deleting an existing key and inserting it again gives it a new insertion position.

In [25]:
def last_wins_and_moves(
    pairs: Iterable[tuple[Hashable, Any]],
) -> dict[Hashable, Any]:
    result: dict[Hashable, Any] = {}

    for key, value in pairs:
        if key in result:
            del result[key]
        result[key] = value

    return result

In [26]:
moved = last_wins_and_moves(observations)
moved

{'pear': 7, 'banana': 3, 'apple': 9, 'kiwi': 4}

## Step 3 — Compare the Three Policies

In [27]:
print("ordinary dict:       ", dict(observations))
print("first wins:          ", first_wins(observations))
print("last wins and moves: ", moved)

ordinary dict:        {'pear': 7, 'apple': 9, 'banana': 3, 'kiwi': 4}
first wins:           {'pear': 2, 'apple': 5, 'banana': 3, 'kiwi': 4}
last wins and moves:  {'pear': 7, 'banana': 3, 'apple': 9, 'kiwi': 4}


In [28]:
assert list(first_wins(observations)) == ["pear", "apple", "banana", "kiwi"]
assert first_wins(observations)["pear"] == 2

assert list(moved) == ["pear", "banana", "apple", "kiwi"]
assert moved["pear"] == 7
assert moved["apple"] == 9

### Best-practice observation

Do not describe duplicate handling merely as "dictionary behavior."

State the full contract:

- which value wins,
- and whether the final order represents first appearance or last appearance.

# Problem 4 — Work Safely with Live Dictionary Views

The objects returned by `keys()`, `values()`, and `items()` are views.

They are not independent lists.

In [29]:
inventory = {"pen": 12, "book": 5}
key_view = inventory.keys()
item_view = inventory.items()

key_view, item_view

(dict_keys(['pen', 'book']), dict_items([('pen', 12), ('book', 5)]))

If the dictionary changes, the views reflect that change.

In [30]:
inventory["eraser"] = 8

key_view, item_view

(dict_keys(['pen', 'book', 'eraser']),
 dict_items([('pen', 12), ('book', 5), ('eraser', 8)]))

This is useful, but it can surprise code that expected a fixed snapshot.

### Your task

Given two configuration dictionaries, produce a report containing:

- keys present in both,
- keys only in the first,
- keys only in the second,
- keys whose values differ,
- and fixed snapshots that do not change when the originals are later modified.

## Step 1 — Dictionary Key Views Support Set-Like Operations

In [31]:
base = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "workers": 2,
}

override = {
    "port": 9000,
    "debug": True,
    "timeout": 30,
}

In [32]:
base.keys() & override.keys()

{'debug', 'port'}

In [33]:
base.keys() - override.keys()

{'host', 'workers'}

## Step 2 — Build the Comparison Report

We will sort set-derived keys so the report is deterministic.

In [34]:
def compare_mappings(
    left: Mapping[Hashable, Any],
    right: Mapping[Hashable, Any],
) -> dict[str, Any]:
    common = left.keys() & right.keys()

    return {
        "common_keys": tuple(sorted(common, key=repr)),
        "left_only": tuple(sorted(left.keys() - right.keys(), key=repr)),
        "right_only": tuple(sorted(right.keys() - left.keys(), key=repr)),
        "changed": {
            key: (left[key], right[key])
            for key in sorted(common, key=repr)
            if left[key] != right[key]
        },
        "left_snapshot": tuple(left.items()),
        "right_snapshot": tuple(right.items()),
    }

In [35]:
report = compare_mappings(base, override)
report

{'common_keys': ('debug', 'port'),
 'left_only': ('host', 'workers'),
 'right_only': ('timeout',),
 'changed': {'debug': (False, True), 'port': (8000, 9000)},
 'left_snapshot': (('host', 'localhost'),
  ('port', 8000),
  ('debug', False),
  ('workers', 2)),
 'right_snapshot': (('port', 9000), ('debug', True), ('timeout', 30))}

## Step 3 — Prove That the Snapshots Are Fixed

In [36]:
base["new_key"] = "later"

print("live keys now:       ", tuple(base.keys()))
print("stored snapshot:     ", report["left_snapshot"])

live keys now:        ('host', 'port', 'debug', 'workers', 'new_key')
stored snapshot:      (('host', 'localhost'), ('port', 8000), ('debug', False), ('workers', 2))


In [37]:
assert "new_key" in base
assert ("new_key", "later") not in report["left_snapshot"]
assert report["changed"] == {
    "debug": (False, True),
    "port": (8000, 9000),
}

### Best-practice observation

Use views for live membership tests and set algebra.

Convert to a tuple or list when you need a stable historical snapshot.

# Problem 5 — Design Composite Keys for Time-Series Measurements

A dictionary key can be a tuple when every element of the tuple is hashable.

In [38]:
sample_key = ("sensor-01", "2026-08-02", 13)
hash(sample_key)

8589492060298992963

Composite keys can represent coordinates in a logical data cube.

For example:

`(sensor_id, day, hour) -> measurement`

### Your task

Build a measurement index that:

- uses a three-part tuple key,
- rejects duplicate measurements for the same key,
- validates hour values,
- supports selecting all measurements for one sensor and day,
- and computes an hourly average across sensors.

## Step 1 — Prepare Sample Records

In [39]:
measurements = [
    {"sensor": "s1", "day": "2026-08-01", "hour": 9, "value": 21.5},
    {"sensor": "s2", "day": "2026-08-01", "hour": 9, "value": 22.5},
    {"sensor": "s1", "day": "2026-08-01", "hour": 10, "value": 22.0},
    {"sensor": "s2", "day": "2026-08-01", "hour": 10, "value": 23.0},
    {"sensor": "s1", "day": "2026-08-02", "hour": 9, "value": 20.0},
]

## Step 2 — Build the Index with Validation

In [40]:
MeasurementKey = tuple[str, str, int]

def build_measurement_index(
    records: Iterable[Mapping[str, Any]],
) -> dict[MeasurementKey, float]:
    index: dict[MeasurementKey, float] = {}

    for position, record in enumerate(records):
        hour = record["hour"]

        if not isinstance(hour, int) or not 0 <= hour <= 23:
            raise ValueError(f"invalid hour at position {position}: {hour!r}")

        key: MeasurementKey = (
            str(record["sensor"]),
            str(record["day"]),
            hour,
        )

        if key in index:
            raise ValueError(f"duplicate measurement key: {key!r}")

        index[key] = float(record["value"])

    return index

In [41]:
measurement_index = build_measurement_index(measurements)
measurement_index

{('s1', '2026-08-01', 9): 21.5,
 ('s2', '2026-08-01', 9): 22.5,
 ('s1', '2026-08-01', 10): 22.0,
 ('s2', '2026-08-01', 10): 23.0,
 ('s1', '2026-08-02', 9): 20.0}

## Step 3 — Select a Sensor-Day Slice

A comprehension is suitable because this is a direct filter and transformation.

In [42]:
def sensor_day_slice(
    index: Mapping[MeasurementKey, float],
    sensor: str,
    day: str,
) -> dict[int, float]:
    return {
        hour: value
        for (record_sensor, record_day, hour), value in index.items()
        if record_sensor == sensor and record_day == day
    }

In [43]:
sensor_day_slice(measurement_index, "s1", "2026-08-01")

{9: 21.5, 10: 22.0}

## Step 4 — Compute Hourly Averages

We first collect totals and counts by `(day, hour)`.

In [44]:
def hourly_averages(
    index: Mapping[MeasurementKey, float],
) -> dict[tuple[str, int], float]:
    totals: dict[tuple[str, int], float] = {}
    counts: dict[tuple[str, int], int] = {}

    for (_, day, hour), value in index.items():
        bucket = (day, hour)
        totals[bucket] = totals.get(bucket, 0.0) + value
        counts[bucket] = counts.get(bucket, 0) + 1

    return {
        bucket: totals[bucket] / counts[bucket]
        for bucket in totals
    }

In [45]:
averages = hourly_averages(measurement_index)
averages

{('2026-08-01', 9): 22.0, ('2026-08-01', 10): 22.5, ('2026-08-02', 9): 20.0}

In [46]:
assert averages[("2026-08-01", 9)] == 22.0
assert averages[("2026-08-01", 10)] == 22.5

### Best-practice observation

Tuple keys are excellent for fixed, positional dimensions.

When the meaning of each position becomes hard to remember, a named immutable key type can be clearer. That is the subject of the next problem.

# Problem 6 — Use a Frozen Dataclass as a Self-Documenting Dictionary Key

A frozen dataclass can be hashable when its fields are hashable.

In [47]:
@dataclass(frozen=True)
class ProductKey:
    store: str
    sku: str
    currency: str

In [48]:
key = ProductKey(store="sofia", sku="A-100", currency="BGN")
hash(key)

8173100023464069797

Compared with a tuple, the fields have names.

That reduces positional mistakes.

### Your task

Create a price book whose keys are `ProductKey` objects.

Then:

- look up equivalent keys,
- demonstrate equality-based lookup,
- group prices by currency,
- and show why a non-frozen dataclass is unsuitable as a dictionary key by default.

## Step 1 — Construct the Price Book

In [49]:
price_book = {
    ProductKey("sofia", "A-100", "BGN"): 12.50,
    ProductKey("sofia", "B-200", "BGN"): 8.75,
    ProductKey("berlin", "A-100", "EUR"): 6.40,
    ProductKey("paris", "C-300", "EUR"): 11.20,
}

price_book

{ProductKey(store='sofia', sku='A-100', currency='BGN'): 12.5,
 ProductKey(store='sofia', sku='B-200', currency='BGN'): 8.75,
 ProductKey(store='berlin', sku='A-100', currency='EUR'): 6.4,
 ProductKey(store='paris', sku='C-300', currency='EUR'): 11.2}

## Step 2 — Look Up with a Newly Constructed but Equal Key

In [50]:
lookup_key = ProductKey("sofia", "A-100", "BGN")

print("same object:", lookup_key is next(iter(price_book)))
print("equal value:", lookup_key == ProductKey("sofia", "A-100", "BGN"))
print("price:", price_book[lookup_key])

same object: False
equal value: True
price: 12.5


Dictionary lookup is based on compatible hashes and equality, not object identity.

## Step 3 — Group Prices by Currency

In [51]:
def prices_by_currency(
    book: Mapping[ProductKey, float],
) -> dict[str, dict[ProductKey, float]]:
    grouped: dict[str, dict[ProductKey, float]] = {}

    for product_key, price in book.items():
        grouped.setdefault(product_key.currency, {})[product_key] = price

    return grouped

In [52]:
grouped_prices = prices_by_currency(price_book)

for currency, entries in grouped_prices.items():
    print(currency, "->", len(entries), "entries")

BGN -> 2 entries
EUR -> 2 entries


## Step 4 — Inspect a Mutable Dataclass

In [53]:
@dataclass
class MutableProductKey:
    store: str
    sku: str

In [54]:
try:
    hash(MutableProductKey("sofia", "A-100"))
except TypeError as exc:
    print(type(exc).__name__, exc)

TypeError unhashable type: 'MutableProductKey'


In [55]:
assert price_book[ProductKey("sofia", "A-100", "BGN")] == 12.50
assert set(grouped_prices) == {"BGN", "EUR"}

### Best-practice observation

Use tuples for compact fixed-position keys.

Use frozen dataclasses when field names improve readability, validation, or domain meaning.

# Problem 7 — Build a Bidirectional Mapping with Uniqueness Enforcement

Sometimes we need fast lookup in both directions.

For example:

- employee ID -> email,
- email -> employee ID.

A single dictionary cannot directly optimize both lookups.

We can build two synchronized dictionaries.

### Your task

Write `build_bimap(pairs)` that returns `(forward, reverse)` and rejects:

- a repeated left key with a different right value,
- a repeated right value with a different left key,
- and unhashable entries.

## Step 1 — Decide the Invariant

For every pair `(left, right)`:

```python
forward[left] == right
reverse[right] == left
```

That means both sides must be unique.

In [56]:
def build_bimap(
    pairs: Iterable[tuple[Hashable, Hashable]],
) -> tuple[dict[Hashable, Hashable], dict[Hashable, Hashable]]:
    forward: dict[Hashable, Hashable] = {}
    reverse: dict[Hashable, Hashable] = {}

    for position, (left, right) in enumerate(pairs):
        try:
            hash(left)
            hash(right)
        except TypeError as exc:
            raise TypeError(
                f"both values must be hashable at position {position}"
            ) from exc

        if left in forward and forward[left] != right:
            raise ValueError(
                f"left value {left!r} is already mapped to {forward[left]!r}"
            )

        if right in reverse and reverse[right] != left:
            raise ValueError(
                f"right value {right!r} is already mapped to {reverse[right]!r}"
            )

        forward[left] = right
        reverse[right] = left

    return forward, reverse

## Step 2 — Build and Query the Mapping

In [57]:
employees = [
    (101, "ada@example.com"),
    (102, "grace@example.com"),
    (103, "linus@example.com"),
]

id_to_email, email_to_id = build_bimap(employees)

print(id_to_email[102])
print(email_to_id["linus@example.com"])

grace@example.com
103


## Step 3 — Demonstrate a Collision

In [58]:
try:
    build_bimap([
        (101, "ada@example.com"),
        (102, "ada@example.com"),
    ])
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError right value 'ada@example.com' is already mapped to 101


In [59]:
assert email_to_id[id_to_email[101]] == 101
assert id_to_email[email_to_id["grace@example.com"]] == "grace@example.com"

### Best-practice observation

When several dictionaries represent one logical structure, construct and validate them in one function.

Do not build the reverse dictionary later with a comprehension unless value uniqueness has already been guaranteed.

# Problem 8 — Parse Repeated Query Parameters into a Multi-Value Dictionary

A web-style query string may repeat the same key.

In [60]:
query = "tag=python&tag=dictionaries&sort=recent&tag=advanced&empty="
parse_qsl(query, keep_blank_values=True)

[('tag', 'python'),
 ('tag', 'dictionaries'),
 ('sort', 'recent'),
 ('tag', 'advanced'),
 ('empty', '')]

Calling `dict(...)` on those pairs loses all but the last value for a repeated key.

In [61]:
dict(parse_qsl(query, keep_blank_values=True))

{'tag': 'advanced', 'sort': 'recent', 'empty': ''}

### Your task

Create a parser that returns:

```python
key -> list of all values in encounter order
```

Then add a normalization step:

- keys are lowercased,
- surrounding whitespace in values is removed,
- blank values may optionally be discarded.

## Step 1 — Group Repeated Keys

In [62]:
def group_query_values(
    query_string: str,
    *,
    discard_blank: bool = False,
) -> dict[str, list[str]]:
    grouped: dict[str, list[str]] = {}

    for raw_key, raw_value in parse_qsl(
        query_string,
        keep_blank_values=True,
    ):
        key = raw_key.strip().lower()
        value = raw_value.strip()

        if discard_blank and value == "":
            continue

        grouped.setdefault(key, []).append(value)

    return grouped

In [63]:
grouped_query = group_query_values(query)
grouped_query

{'tag': ['python', 'dictionaries', 'advanced'],
 'sort': ['recent'],
 'empty': ['']}

## Step 2 — Derive Single-Valued and Multi-Valued Views

Some applications want a convenient scalar only when a key occurs once.

In [64]:
def split_query_cardinality(
    grouped: Mapping[str, Sequence[str]],
) -> dict[str, dict[str, Any]]:
    singles: dict[str, str] = {}
    multiples: dict[str, tuple[str, ...]] = {}

    for key, values in grouped.items():
        if len(values) == 1:
            singles[key] = values[0]
        else:
            multiples[key] = tuple(values)

    return {
        "single": singles,
        "multiple": multiples,
    }

In [65]:
split_query_cardinality(grouped_query)

{'single': {'sort': 'recent', 'empty': ''},
 'multiple': {'tag': ('python', 'dictionaries', 'advanced')}}

## Step 3 — Verify Ordering and Blank Handling

In [66]:
assert grouped_query["tag"] == ["python", "dictionaries", "advanced"]
print(
    "Is the retained blank represented by an empty list?",
    grouped_query["empty"] == [],
)

Is the retained blank represented by an empty list? False


The previous assertion is intentionally wrong for our current contract: blank values are retained as empty strings.

Let us inspect and then assert the correct result.

In [67]:
print("retained blank value:", grouped_query["empty"])
assert grouped_query["empty"] == [""]
assert "empty" not in group_query_values(query, discard_blank=True)

retained blank value: ['']


### Best-practice observation

A dictionary from keys to lists is a natural representation for repeated-key data.

Document whether empty values are kept and whether value order matters.

# Problem 9 — Build Primary and Secondary Indexes for Records

Suppose records are stored in a list.

In [68]:
products = [
    {"id": 1, "category": "book", "supplier": "north", "name": "Python Patterns"},
    {"id": 2, "category": "book", "supplier": "south", "name": "Data Structures"},
    {"id": 3, "category": "tool", "supplier": "north", "name": "Keyboard"},
    {"id": 4, "category": "tool", "supplier": "east", "name": "Monitor Stand"},
    {"id": 5, "category": "book", "supplier": "north", "name": "Algorithms"},
]

Scanning the whole list for every query costs time and repeats logic.

We can build:

- a primary index: `id -> record`,
- a secondary index: `category -> tuple of ids`,
- another secondary index: `supplier -> tuple of ids`.

### Your task

Write a reusable `build_indexes` function that:

- rejects duplicate primary keys,
- preserves record order inside secondary groups,
- stores immutable tuples in the final secondary indexes,
- and does not copy each record unnecessarily.

## Step 1 — Build Mutable Buckets Internally

Lists are convenient while accumulating IDs.

In [69]:
def build_indexes(
    records: Iterable[Mapping[str, Any]],
    *,
    primary_field: str,
    secondary_fields: Sequence[str],
) -> tuple[
    dict[Hashable, Mapping[str, Any]],
    dict[str, dict[Hashable, tuple[Hashable, ...]]],
]:
    primary: dict[Hashable, Mapping[str, Any]] = {}
    mutable_secondary: dict[
        str,
        dict[Hashable, list[Hashable]],
    ] = {
        field: {}
        for field in secondary_fields
    }

    for position, record in enumerate(records):
        primary_key = record[primary_field]

        if primary_key in primary:
            raise ValueError(
                f"duplicate {primary_field!r} at position {position}: "
                f"{primary_key!r}"
            )

        primary[primary_key] = record

        for field in secondary_fields:
            field_value = record[field]
            mutable_secondary[field].setdefault(field_value, []).append(primary_key)

    secondary = {
        field: {
            value: tuple(primary_keys)
            for value, primary_keys in groups.items()
        }
        for field, groups in mutable_secondary.items()
    }

    return primary, secondary

## Step 2 — Construct the Indexes

In [70]:
product_by_id, product_indexes = build_indexes(
    products,
    primary_field="id",
    secondary_fields=("category", "supplier"),
)

product_by_id

{1: {'id': 1,
  'category': 'book',
  'supplier': 'north',
  'name': 'Python Patterns'},
 2: {'id': 2,
  'category': 'book',
  'supplier': 'south',
  'name': 'Data Structures'},
 3: {'id': 3, 'category': 'tool', 'supplier': 'north', 'name': 'Keyboard'},
 4: {'id': 4, 'category': 'tool', 'supplier': 'east', 'name': 'Monitor Stand'},
 5: {'id': 5, 'category': 'book', 'supplier': 'north', 'name': 'Algorithms'}}

In [71]:
product_indexes

{'category': {'book': (1, 2, 5), 'tool': (3, 4)},
 'supplier': {'north': (1, 3, 5), 'south': (2,), 'east': (4,)}}

## Step 3 — Query Through an Index

First retrieve matching IDs, then use the primary index to access the records.

In [72]:
north_product_ids = product_indexes["supplier"]["north"]
north_products = [product_by_id[product_id] for product_id in north_product_ids]

north_products

[{'id': 1, 'category': 'book', 'supplier': 'north', 'name': 'Python Patterns'},
 {'id': 3, 'category': 'tool', 'supplier': 'north', 'name': 'Keyboard'},
 {'id': 5, 'category': 'book', 'supplier': 'north', 'name': 'Algorithms'}]

In [73]:
assert north_product_ids == (1, 3, 5)
assert product_by_id[1] is products[0]

### Best-practice observation

Secondary indexes usually store primary keys rather than duplicate copies of full records.

That reduces memory usage and gives one authoritative record object.

# Problem 10 — Create a Callable Dispatch Table with Preconfigured Operations

Functions are objects and can be stored as dictionary values.

In [74]:
def add(a: float, b: float) -> float:
    return a + b

def multiply(a: float, b: float) -> float:
    return a * b

In [75]:
operations = {
    "add": add,
    "multiply": multiply,
}

operations["add"](3, 4)

7

A dispatch table can replace a long `if` / `elif` chain.

`functools.partial` can preconfigure arguments while still producing a callable.

### Your task

Build a pricing-operation registry supporting:

- percentage discount,
- fixed discount,
- tax,
- and a no-op operation.

Each operation should accept a single current price.

Then write a pipeline runner that validates operation names and rejects negative intermediate prices.

## Step 1 — Define General-Purpose Functions

In [76]:
def percentage_discount(price: float, *, rate: float) -> float:
    return price * (1.0 - rate)

def fixed_discount(price: float, *, amount: float) -> float:
    return price - amount

def add_tax(price: float, *, rate: float) -> float:
    return price * (1.0 + rate)

def unchanged(price: float) -> float:
    return price

## Step 2 — Build the Registry with Preconfigured Callables

In [77]:
pricing_operations: dict[str, Callable[[float], float]] = {
    "student_discount": partial(percentage_discount, rate=0.10),
    "coupon_5": partial(fixed_discount, amount=5.0),
    "vat_20": partial(add_tax, rate=0.20),
    "none": unchanged,
}

In [78]:
pricing_operations["student_discount"](100.0)

90.0

## Step 3 — Run a Named Pipeline

In [79]:
def run_price_pipeline(
    starting_price: float,
    operation_names: Iterable[str],
    registry: Mapping[str, Callable[[float], float]],
) -> dict[str, Any]:
    current = float(starting_price)
    history: list[dict[str, float | str]] = []

    for step, name in enumerate(operation_names, start=1):
        if name not in registry:
            raise KeyError(f"unknown pricing operation: {name!r}")

        before = current
        current = float(registry[name](current))

        if current < 0:
            raise ValueError(
                f"operation {name!r} produced a negative price at step {step}"
            )

        history.append({
            "operation": name,
            "before": before,
            "after": current,
        })

    return {
        "starting_price": float(starting_price),
        "final_price": current,
        "history": history,
    }

In [80]:
pricing_result = run_price_pipeline(
    100.0,
    ["student_discount", "coupon_5", "vat_20"],
    pricing_operations,
)

pricing_result

{'starting_price': 100.0,
 'final_price': 102.0,
 'history': [{'operation': 'student_discount', 'before': 100.0, 'after': 90.0},
  {'operation': 'coupon_5', 'before': 90.0, 'after': 85.0},
  {'operation': 'vat_20', 'before': 85.0, 'after': 102.0}]}

In [81]:
assert round(pricing_result["final_price"], 2) == 102.00

### Best-practice observation

Use a registry when operation names are data.

Validate names before invocation and keep the callable contract consistent across all registered functions.

# Problem 11 — Perform a Copy-on-Write Update on a Nested Dictionary

A shallow copy creates a new outer dictionary but reuses nested objects.

In [82]:
original_settings = {
    "database": {
        "host": "localhost",
        "credentials": {
            "user": "reader",
            "password": "secret",
        },
    },
    "features": {
        "search": True,
    },
}

shallow = dict(original_settings)

print(original_settings is shallow)
print(original_settings["database"] is shallow["database"])

False
True


Sometimes a full deep copy is more copying than we need.

A **copy-on-write path update** copies only dictionaries along the modified path.

### Your task

Implement:

```python
set_path_copy(mapping, path, value)
```

It should:

- return a new nested dictionary,
- leave the original unchanged,
- copy only dictionaries along the selected path,
- preserve untouched branches by identity,
- and reject an empty path.

## Step 1 — Define the Recursive Case

At each path segment:

1. copy the current dictionary,
2. recursively replace one child,
3. return the new dictionary.

In [83]:
def set_path_copy(
    mapping: Mapping[Hashable, Any],
    path: Sequence[Hashable],
    value: Any,
) -> dict[Hashable, Any]:
    if not path:
        raise ValueError("path must contain at least one key")

    key = path[0]
    result = dict(mapping)

    if len(path) == 1:
        result[key] = value
        return result

    existing_child = mapping.get(key, {})

    if not isinstance(existing_child, Mapping):
        raise TypeError(
            f"cannot descend through non-mapping value at key {key!r}"
        )

    result[key] = set_path_copy(existing_child, path[1:], value)
    return result

## Step 2 — Update a Deep Value

In [84]:
updated_settings = set_path_copy(
    original_settings,
    ("database", "credentials", "password"),
    "new-secret",
)

updated_settings

{'database': {'host': 'localhost',
  'credentials': {'user': 'reader', 'password': 'new-secret'}},
 'features': {'search': True}}

## Step 3 — Inspect Equality and Identity

The modified path receives new dictionary objects.

The untouched `"features"` branch is shared safely because it was not mutated.

In [85]:
print("outer copied:", original_settings is not updated_settings)
print(
    "database copied:",
    original_settings["database"] is not updated_settings["database"],
)
print(
    "credentials copied:",
    original_settings["database"]["credentials"]
    is not updated_settings["database"]["credentials"],
)
print(
    "features shared:",
    original_settings["features"] is updated_settings["features"],
)

outer copied: True
database copied: True
credentials copied: True
features shared: True


In [86]:
assert original_settings["database"]["credentials"]["password"] == "secret"
assert updated_settings["database"]["credentials"]["password"] == "new-secret"
assert original_settings["features"] is updated_settings["features"]

### Best-practice observation

Copy-on-write is useful when nested structures are treated as immutable after construction.

If callers may mutate shared untouched branches later, use a deep copy or stronger immutable structures instead.

# Problem 12 — Build a Nested Cross-Tabulation Dictionary

A cross-tabulation summarizes records across two dimensions.

For example:

```python
department -> quarter -> total revenue
```

In [87]:
sales = [
    {"department": "books", "quarter": "Q1", "amount": 1200},
    {"department": "tools", "quarter": "Q1", "amount": 900},
    {"department": "books", "quarter": "Q2", "amount": 1500},
    {"department": "tools", "quarter": "Q2", "amount": 1100},
    {"department": "books", "quarter": "Q1", "amount": 300},
    {"department": "games", "quarter": "Q2", "amount": 700},
]

### Your task

Build a nested dictionary with totals, then normalize it so every department contains every observed quarter with a default of zero.

Avoid the shared-mutable-value mistake that can occur with `dict.fromkeys`.

## Step 1 — Aggregate Only the Observed Combinations

In [88]:
def cross_tabulate(
    records: Iterable[Mapping[str, Any]],
    *,
    row_field: str,
    column_field: str,
    value_field: str,
) -> dict[Hashable, dict[Hashable, float]]:
    table: dict[Hashable, dict[Hashable, float]] = {}

    for record in records:
        row = record[row_field]
        column = record[column_field]
        amount = float(record[value_field])

        row_dict = table.setdefault(row, {})
        row_dict[column] = row_dict.get(column, 0.0) + amount

    return table

In [89]:
observed_table = cross_tabulate(
    sales,
    row_field="department",
    column_field="quarter",
    value_field="amount",
)

observed_table

{'books': {'Q1': 1500.0, 'Q2': 1500.0},
 'tools': {'Q1': 900.0, 'Q2': 1100.0},
 'games': {'Q2': 700.0}}

## Step 2 — Normalize Missing Columns

A new inner dictionary must be constructed for every row.

In [90]:
def normalize_columns(
    table: Mapping[Hashable, Mapping[Hashable, float]],
    columns: Iterable[Hashable],
    *,
    default: float = 0.0,
) -> dict[Hashable, dict[Hashable, float]]:
    column_tuple = tuple(columns)

    return {
        row: {
            column: values.get(column, default)
            for column in column_tuple
        }
        for row, values in table.items()
    }

In [91]:
all_quarters = ("Q1", "Q2")
normalized_table = normalize_columns(observed_table, all_quarters)
normalized_table

{'books': {'Q1': 1500.0, 'Q2': 1500.0},
 'tools': {'Q1': 900.0, 'Q2': 1100.0},
 'games': {'Q1': 0.0, 'Q2': 700.0}}

## Step 3 — Confirm Inner Dictionaries Are Independent

In [92]:
print(
    normalized_table["books"] is normalized_table["tools"],
    normalized_table["tools"] is normalized_table["games"],
)

False False


In [93]:
assert normalized_table["books"] == {"Q1": 1500.0, "Q2": 1500.0}
assert normalized_table["games"] == {"Q1": 0.0, "Q2": 700.0}
assert normalized_table["books"] is not normalized_table["tools"]

### Best-practice observation

Use `dict.fromkeys(columns, immutable_default)` safely for immutable defaults.

For mutable nested values, create a fresh object per key with a comprehension or loop.

# Problem 13 — Create a Lazy Dictionary with `__missing__`

A normal dictionary raises `KeyError` when a key is absent.

In [94]:
ordinary_cache = {}

try:
    ordinary_cache[5]
except KeyError as exc:
    print(type(exc).__name__, exc)

KeyError 5


A dictionary subclass may define `__missing__(self, key)`.

Python calls it for `mapping[key]` when the key is absent.

### Your task

Create `SquareCache`, a dictionary that:

- accepts non-negative integers as keys,
- computes and stores `key ** 2` on first access,
- returns the stored value on later access,
- and rejects booleans, negative integers, and non-integers.

## Step 1 — Define the Subclass

In [95]:
class SquareCache(dict[int, int]):
    def __missing__(self, key: int) -> int:
        if isinstance(key, bool) or not isinstance(key, int):
            raise TypeError("SquareCache keys must be integers")

        if key < 0:
            raise ValueError("SquareCache keys must be non-negative")

        value = key * key
        self[key] = value
        return value

## Step 2 — Observe Lazy Insertion

In [96]:
squares = SquareCache()

print("before:", squares)
print("lookup:", squares[12])
print("after: ", squares)

before: {}
lookup: 144
after:  {12: 144}


The second lookup uses the stored value. `__missing__` is not called again.

In [97]:
assert squares[12] == 144
assert squares == {12: 144}

## Step 3 — Compare `[]` and `.get()`

The `dict.get` method does not invoke `__missing__`.

In [98]:
print("get result:", squares.get(7))
print("after get:", squares)
print("indexed result:", squares[7])
print("after indexing:", squares)

get result: None
after get: {12: 144}
indexed result: 49
after indexing: {12: 144, 7: 49}


In [99]:
try:
    squares[-1]
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError SquareCache keys must be non-negative


### Best-practice observation

Use `__missing__` when missing-key computation is part of the mapping's abstraction.

Do not use it merely to hide unexpected missing data.

# Problem 14 — Perform a Hash Join Between Two Collections of Records

A join combines records that share a key.

Suppose we have customers and orders.

In [100]:
customers = [
    {"customer_id": 1, "name": "Ada", "tier": "gold"},
    {"customer_id": 2, "name": "Grace", "tier": "silver"},
    {"customer_id": 3, "name": "Linus", "tier": "gold"},
]

orders = [
    {"order_id": "A1", "customer_id": 2, "total": 80.0},
    {"order_id": "A2", "customer_id": 1, "total": 120.0},
    {"order_id": "A3", "customer_id": 2, "total": 40.0},
    {"order_id": "A4", "customer_id": 99, "total": 25.0},
]

A nested-loop join compares every order with every customer.

A hash join first builds a dictionary index, then performs constant-time average lookups.

### Your task

Implement a left join from orders to customers.

Each result should contain the order fields plus:

- `customer_name`,
- `customer_tier`,
- and `matched_customer`.

Unmatched orders must remain in the output.

## Step 1 — Build a Unique Customer Index

In [101]:
def unique_index(
    records: Iterable[Mapping[str, Any]],
    field: str,
) -> dict[Hashable, Mapping[str, Any]]:
    index: dict[Hashable, Mapping[str, Any]] = {}

    for position, record in enumerate(records):
        key = record[field]

        if key in index:
            raise ValueError(
                f"duplicate join key {key!r} at position {position}"
            )

        index[key] = record

    return index

In [102]:
customer_index = unique_index(customers, "customer_id")
customer_index

{1: {'customer_id': 1, 'name': 'Ada', 'tier': 'gold'},
 2: {'customer_id': 2, 'name': 'Grace', 'tier': 'silver'},
 3: {'customer_id': 3, 'name': 'Linus', 'tier': 'gold'}}

## Step 2 — Join Each Order Against the Index

In [103]:
def left_join_orders_to_customers(
    order_records: Iterable[Mapping[str, Any]],
    customer_records: Iterable[Mapping[str, Any]],
) -> list[dict[str, Any]]:
    customer_by_id = unique_index(customer_records, "customer_id")
    joined: list[dict[str, Any]] = []

    for order in order_records:
        customer = customer_by_id.get(order["customer_id"])

        joined.append({
            **order,
            "customer_name": None if customer is None else customer["name"],
            "customer_tier": None if customer is None else customer["tier"],
            "matched_customer": customer is not None,
        })

    return joined

In [104]:
joined_orders = left_join_orders_to_customers(orders, customers)
joined_orders

[{'order_id': 'A1',
  'customer_id': 2,
  'total': 80.0,
  'customer_name': 'Grace',
  'customer_tier': 'silver',
  'matched_customer': True},
 {'order_id': 'A2',
  'customer_id': 1,
  'total': 120.0,
  'customer_name': 'Ada',
  'customer_tier': 'gold',
  'matched_customer': True},
 {'order_id': 'A3',
  'customer_id': 2,
  'total': 40.0,
  'customer_name': 'Grace',
  'customer_tier': 'silver',
  'matched_customer': True},
 {'order_id': 'A4',
  'customer_id': 99,
  'total': 25.0,
  'customer_name': None,
  'customer_tier': None,
  'matched_customer': False}]

## Step 3 — Summarize Joined and Unmatched Orders

In [105]:
join_summary = {
    "matched": sum(record["matched_customer"] for record in joined_orders),
    "unmatched_order_ids": tuple(
        record["order_id"]
        for record in joined_orders
        if not record["matched_customer"]
    ),
}

join_summary

{'matched': 3, 'unmatched_order_ids': ('A4',)}

In [106]:
assert join_summary == {
    "matched": 3,
    "unmatched_order_ids": ("A4",),
}

### Best-practice observation

Build the index on the side whose join keys must be unique.

Make unmatched behavior explicit rather than silently dropping records.

# Problem 15 — Capstone: Build a Dictionary-Driven Rule Engine with an Audit Trail

We will combine several ideas from the notebook.

A rule engine receives a record and applies named rules.

Each rule has:

- a predicate,
- an action,
- a priority,
- and a human-readable description.

We will represent the rule registry as:

```python
rule_name -> rule specification dictionary
```

The engine will:

- validate the registry,
- evaluate rules in priority order,
- preserve registry insertion order for equal priorities,
- apply matching actions,
- and produce a detailed audit trail.

## Step 1 — Define Predicate and Action Functions

Predicates inspect a record.

Actions return a new record instead of mutating the old one.

In [107]:
Record = dict[str, Any]
Predicate = Callable[[Mapping[str, Any]], bool]
Action = Callable[[Mapping[str, Any]], Record]

def is_large_order(record: Mapping[str, Any]) -> bool:
    return float(record["total"]) >= 100.0

def is_gold_customer(record: Mapping[str, Any]) -> bool:
    return record.get("tier") == "gold"

def is_international(record: Mapping[str, Any]) -> bool:
    return record.get("country") != "BG"

def apply_large_order_discount(record: Mapping[str, Any]) -> Record:
    return {
        **record,
        "total": round(float(record["total"]) * 0.95, 2),
    }

def add_gold_points(record: Mapping[str, Any]) -> Record:
    return {
        **record,
        "points": int(record.get("points", 0)) + 100,
    }

def add_international_fee(record: Mapping[str, Any]) -> Record:
    return {
        **record,
        "total": round(float(record["total"]) + 12.0, 2),
    }

## Step 2 — Construct the Rule Registry

In [108]:
rule_registry = {
    "large_order_discount": {
        "predicate": is_large_order,
        "action": apply_large_order_discount,
        "priority": 10,
        "description": "Apply a 5% discount to orders of at least 100.",
    },
    "gold_points": {
        "predicate": is_gold_customer,
        "action": add_gold_points,
        "priority": 20,
        "description": "Add 100 loyalty points for a gold customer.",
    },
    "international_fee": {
        "predicate": is_international,
        "action": add_international_fee,
        "priority": 30,
        "description": "Add a fixed fee to international orders.",
    },
}

## Step 3 — Validate the Registry

Validation is easier to read as an explicit loop.

In [109]:
def validate_rule_registry(
    registry: Mapping[str, Mapping[str, Any]],
) -> None:
    required_fields = {
        "predicate",
        "action",
        "priority",
        "description",
    }

    for rule_name, specification in registry.items():
        missing = required_fields - specification.keys()

        if missing:
            raise ValueError(
                f"rule {rule_name!r} is missing fields: {sorted(missing)!r}"
            )

        if not callable(specification["predicate"]):
            raise TypeError(f"predicate for {rule_name!r} must be callable")

        if not callable(specification["action"]):
            raise TypeError(f"action for {rule_name!r} must be callable")

        if not isinstance(specification["priority"], int):
            raise TypeError(f"priority for {rule_name!r} must be an integer")

In [110]:
validate_rule_registry(rule_registry)
print("registry is valid")

registry is valid


## Step 4 — Order the Rules

Python's sort is stable.

Therefore, rules with equal priorities retain the registry's insertion order.

In [111]:
def ordered_rules(
    registry: Mapping[str, Mapping[str, Any]],
) -> list[tuple[str, Mapping[str, Any]]]:
    return sorted(
        registry.items(),
        key=lambda item: item[1]["priority"],
    )

In [112]:
[(name, spec["priority"]) for name, spec in ordered_rules(rule_registry)]

[('large_order_discount', 10), ('gold_points', 20), ('international_fee', 30)]

## Step 5 — Run the Engine and Record Every Decision

In [113]:
def run_rule_engine(
    record: Mapping[str, Any],
    registry: Mapping[str, Mapping[str, Any]],
) -> dict[str, Any]:
    validate_rule_registry(registry)

    current: Record = dict(record)
    audit: list[dict[str, Any]] = []

    for rule_name, specification in ordered_rules(registry):
        matched = bool(specification["predicate"](current))
        before = dict(current)

        if matched:
            current = dict(specification["action"](current))

        audit.append({
            "rule": rule_name,
            "priority": specification["priority"],
            "description": specification["description"],
            "matched": matched,
            "before": before,
            "after": dict(current),
        })

    return {
        "input": dict(record),
        "output": current,
        "audit": audit,
    }

## Step 6 — Execute the Capstone Example

In [114]:
order = {
    "order_id": "B-100",
    "total": 140.0,
    "tier": "gold",
    "country": "DE",
    "points": 20,
}

engine_result = run_rule_engine(order, rule_registry)
engine_result["output"]

{'order_id': 'B-100',
 'total': 145.0,
 'tier': 'gold',
 'country': 'DE',
 'points': 120}

Let us display a concise audit summary.

In [115]:
for entry in engine_result["audit"]:
    print(
        f"{entry['priority']:>2} | "
        f"{entry['rule']:<22} | "
        f"matched={entry['matched']}"
    )

10 | large_order_discount   | matched=True
20 | gold_points            | matched=True
30 | international_fee      | matched=True


## Step 7 — Verify Immutability and Final Values

In [116]:
print("original:", order)
print("result:  ", engine_result["output"])

original: {'order_id': 'B-100', 'total': 140.0, 'tier': 'gold', 'country': 'DE', 'points': 20}
result:   {'order_id': 'B-100', 'total': 145.0, 'tier': 'gold', 'country': 'DE', 'points': 120}


In [117]:
assert order == {
    "order_id": "B-100",
    "total": 140.0,
    "tier": "gold",
    "country": "DE",
    "points": 20,
}

assert engine_result["output"] == {
    "order_id": "B-100",
    "total": 145.0,
    "tier": "gold",
    "country": "DE",
    "points": 120,
}

The calculation is:

1. `140.00` receives a 5% discount -> `133.00`,
2. 100 points are added,
3. the international fee adds `12.00` -> `145.00`.

### Capstone best practices

- Registry entries use consistent field names.
- Functions share consistent callable contracts.
- Rule order is deterministic.
- Actions return new dictionaries.
- The input record remains unchanged.
- Every decision is recorded for debugging and auditability.

# Additional Practice Challenges

The notebook already contains complete solutions above.

The following extensions are left as independent practice:

1. Add an `enabled` field to every rule specification.
2. Add a `stop_processing` field that terminates the engine after a matching rule.
3. Add a secondary index from rule priority to rule names.
4. Modify the query parser so selected keys are converted to integers.
5. Extend the hash join to support one customer with many addresses.
6. Add deletion support to the copy-on-write path function.
7. Add a third dimension to the cross-tabulation.
8. Make the lazy dictionary compute Fibonacci numbers instead of squares.
9. Add a consistency checker for the bidirectional map.
10. Serialize frozen dataclass keys into JSON-friendly records.

# Final Review

We created dictionaries using:

- `dict(...)`,
- validated loops,
- comprehensions,
- `setdefault`,
- tuple keys,
- frozen dataclass keys,
- callable registries,
- dictionary subclasses,
- and nested indexing structures.

The main lesson is that advanced dictionary work is usually about **defining the construction contract**:

- what counts as a valid key,
- what duplicate input means,
- what ordering represents,
- whether data is copied or shared,
- and how missing values are handled.

A dictionary is simple to create. A reliable dictionary-based design requires those choices to be explicit.